# Context, concurrency and reasoning

*Local AI Inference Lab: Serving, Optimization and Application Quality*

**Question:** How do we make local AI faster and more efficient without losing useful task performance?

**Learn:** investigate hybrid state, context, concurrency/capacity and reasoning budgets.
**Requirements:** [CPU replay prerequisites](../SETUP.md); [Live configuration](../tutorial/RUN_ALL.md) only for fresh GPU runs.
**Expected outputs:** recorded tables, a server command/request/evaluated SQL example,
and an observation → explanation → limitation → decision for every experiment.

This second chapter applies the same quality-first method to a hybrid model. Read the Qwen chapter for the common mechanics.

These are observations from a small recorded Dell workload, not universal claims. A faster
response is not necessarily a better answer; different optimizations affect different stages.
Clinical-trial analytics supplies an executable application-quality test. We have not evaluated
multiple application domains. The synthetic database contains no patient data.


## 1. Choose how to explore

| Mode | Behavior |
|---|---|
| **Replay — default** | Load checksum-verified **recorded Dell measurements** and reproduce tables without a GPU. |
| Plan | Show configuration and server commands without launching or downloading anything. |
| Live | Launch owned servers, benchmark, preserve evidence and clean up. Requires Linux GPU setup. |

Choose **Run All** now for replay. To change mode, create `tutorial/config/vllm-nemotron.local.json`
using [the shared setup guide](../tutorial/RUN_ALL.md) and set `mode` to `replay`, `plan` or `live`.
All machine settings and flags belong in this configuration file. A live failure never silently
loads archived results. Replay ignores new experiment settings when presenting historical values.

Serving target: **Nemotron 3 Nano 4B**, `nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16`. The checkpoint is loaded as **FP16** through
native vLLM in a separate GPU Python environment.
The hybrid Mamba/attention architecture retains FP32 SSM state. Its BF16 source label does not change the requested FP16 experiment.

Four distinct clinical questions are repeated five times per serving condition. They are not
20 independent clinical tasks. Application quality separately evaluates four cases, including
planning, validation, SQL execution and one repair. Exact-SQL semantic review is still pending.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/clinical_trial_mvp').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import JSON, Markdown, display
from clinical_trial_mvp.tutorial.workflow import Study
from clinical_trial_mvp.tutorial.teaching import server_example, request_example, response_example, interpret
if 'study' in globals():
    study.close()
study = Study('vllm', 'nemotron', root=ROOT)
results = study.results
print('Mode:', study.mode)
print('Model:', 'nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16')
print(study.prepare())


Mode: replay
Model: nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16
Replay: recorded Dell measurements; no model download, server, GPU or network calls.


## 2. Predict before measuring

Recap: serving speed and application quality are different outcomes. Use
[chapter 01](01_qwen_cache_quantization.ipynb) for prefill/decode, W4A16 and common metric
definitions. Here, focus on the model/deployment differences below and predict their effects.


Establish the FP16 baseline, then investigate verified hybrid prefix caching,
retrieved context, concurrency 1/2/4 at scheduler capacity 4, capacity 1 with four clients,
reasoning on, and output caps 256/2048. Client overlap and scheduler capacity are distinct.
Reasoning tokens consume output budget, and TTFT includes the first thought token.

Nemotron core has no AWQ dependency. AWQ is optional stretch work and never gates core runs.
Set `prefix_supported=true` only after verifying the installed hybrid backend. Unsupported
cache/AWQ rows are unavailable or skipped, not zero performance. No compatible Nemotron
AWQ artifact was obtained; its absence does not invalidate completed core experiments.


## 3. Compatibility before benchmarking

Smoke asks whether the path can generate and execute SQL. It is not a general quality score.
Replay loads the recorded smoke when available. Plan makes no model calls. Live stops on
failed required smoke, retains the failure and never substitutes replay measurements.


In [2]:
smoke = study.smoke()
if isinstance(smoke, dict):
    print(smoke.get('_recorded', {}).get('label', study.mode), smoke.get('status'), smoke.get('reason', ''))
else:
    print(smoke)


replay unavailable No such measurement was recorded in the Dell archive; no values estimated.


## 4. Experiments and decisions

Read each prediction before running its cell. Quality counts accompany speed; unavailable metrics stay unavailable.


### Input work: full/retrieved schema and prefix caching

Each row is its own controlled comparison. No cumulative best configuration was measured.


#### fp16 baseline


In [3]:
result = study.run('fp16_baseline')
print(result.get('_recorded', {}).get('label', study.mode), 'fp16_baseline:', result['status'])
display(Markdown(interpret(study, 'fp16_baseline')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'fp16_baseline')))


recorded Dell measurement fp16_baseline: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T002501_223219Z.json](../experiment-tracker/mvp4/nemotron/20261107T002501_223219Z.json)

**Prediction:** Establish executable SQL and separate application quality from request speed.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 4.251/4.892 s; aggregate output: 36.622 tok/s.

**Explanation:** Transport timing and application correctness measure different stages.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Use this baseline to judge useful completed tasks, not latency alone.

### Under the hood: three inspectable paths

**A. Server command.** The next cell shows the recorded portable argv or the planned command,
model revision, dtype, cache control and scheduler limits. A setting absent from the archive
is marked unknown rather than inferred.
[Native launcher](../src/clinical_trial_mvp/serving/launch.py).


In [4]:
display(JSON(server_example(study, 'fp16_baseline')))


<IPython.core.display.JSON object>

**B. One HTTP request.** Inspect the full prompt, generation options and schema
contract. This uses the exact payload builder called by the transport. Replay reconstructs
it from the saved prompt and recorded flags; it is not a packet capture. No request is sent
by this inspection cell. [Request builder and timing](../src/clinical_trial_mvp/serving/client.py).


In [5]:
display(JSON(request_example(study, 'fp16_baseline')))


<IPython.core.display.JSON object>

**C. One evaluated response.** Inspect generated SQL, safety/schema validation,
execution outcome, query rows, the reference query and the recorded result comparison.
A reference match is not a human semantic review. [Plan/SQL validation](../src/clinical_trial_mvp/agent/analyst.py)
· [reference scorer](../src/clinical_trial_mvp/evaluation/scorers.py).


In [6]:
display(JSON(response_example(study, 'fp16_baseline')))


<IPython.core.display.JSON object>

#### prefix cache


In [7]:
result = study.run('prefix_cache')
print(result.get('_recorded', {}).get('label', study.mode), 'prefix_cache:', result['status'])
display(Markdown(interpret(study, 'prefix_cache')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'prefix_cache')))


recorded Dell measurement prefix_cache: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T023148_583457Z.json](../experiment-tracker/mvp4/nemotron/20261107T023148_583457Z.json)

**Prediction:** Shared-prefix reuse should reduce warm prefill latency more than total generation time.

**Observation:** Cold → warm TTFT p50: 1.101 → 0.300 s. Request E2E p50: 4.327 → 3.529 s. Direct SQL result match: 15/20 cold; 15/20 warm.

**Explanation:** The paired phase records are consistent with avoiding repeated prefill; generation still consumes time.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Consider caching for repeated schema/system prefixes after checking both phase quality counts.

#### schema retrieved


In [8]:
result = study.run('schema_retrieved')
print(result.get('_recorded', {}).get('label', study.mode), 'schema_retrieved:', result['status'])
display(Markdown(interpret(study, 'schema_retrieved')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'schema_retrieved')))


recorded Dell measurement schema_retrieved: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T024417_765664Z.json](../experiment-tracker/mvp4/nemotron/20261107T024417_765664Z.json)

**Prediction:** Less context may reduce input work but can remove definitions required for correct SQL.

**Observation:** Application result match: 1/4; SQL executed: 1/4. Request E2E p50/p95: 2.593/3.546 s; aggregate output: 46.300 tok/s. Baseline application result match: 2/4.

**Explanation:** Context reduction changes the information available to the model, not just compute cost.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Retain needed clinical definitions; accept reduced context only with adequate task quality.

### Serving capacity: clients and scheduler limits

Each row is its own controlled comparison. No cumulative best configuration was measured.


#### concurrency 2


In [9]:
result = study.run('concurrency_2')
print(result.get('_recorded', {}).get('label', study.mode), 'concurrency_2:', result['status'])
display(Markdown(interpret(study, 'concurrency_2')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'concurrency_2')))


recorded Dell measurement concurrency_2: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T025050_224975Z.json](../experiment-tracker/mvp4/nemotron/20261107T025050_224975Z.json)

**Prediction:** More overlapping clients may improve aggregate throughput while increasing per-request latency.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 5.362/7.011 s; aggregate output: 56.243 tok/s.

**Explanation:** Client overlap and scheduler capacity are separate controls; these records do not show achieved batch occupancy.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Choose concurrency using both latency limits and correctly completed work.

#### concurrency 4


In [10]:
result = study.run('concurrency_4')
print(result.get('_recorded', {}).get('label', study.mode), 'concurrency_4:', result['status'])
display(Markdown(interpret(study, 'concurrency_4')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'concurrency_4')))


recorded Dell measurement concurrency_4: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T030851_257660Z.json](../experiment-tracker/mvp4/nemotron/20261107T030851_257660Z.json)

**Prediction:** More overlapping clients may improve aggregate throughput while increasing per-request latency.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 7.699/10.557 s; aggregate output: 76.876 tok/s.

**Explanation:** Client overlap and scheduler capacity are separate controls; these records do not show achieved batch occupancy.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Choose concurrency using both latency limits and correctly completed work.

#### batch capacity 1


In [11]:
result = study.run('batch_capacity_1')
print(result.get('_recorded', {}).get('label', study.mode), 'batch_capacity_1:', result['status'])
display(Markdown(interpret(study, 'batch_capacity_1')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'batch_capacity_1')))


recorded Dell measurement batch_capacity_1: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T031845_031809Z.json](../experiment-tracker/mvp4/nemotron/20261107T031845_031809Z.json)

**Prediction:** More overlapping clients may improve aggregate throughput while increasing per-request latency.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 16.704/16.748 s; aggregate output: 36.382 tok/s.

**Explanation:** Client overlap and scheduler capacity are separate controls; these records do not show achieved batch occupancy.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Choose concurrency using both latency limits and correctly completed work.

### Output work: reasoning and token caps

Each row is its own controlled comparison. No cumulative best configuration was measured.


#### reasoning on


In [12]:
result = study.run('reasoning_on')
print(result.get('_recorded', {}).get('label', study.mode), 'reasoning_on:', result['status'])
display(Markdown(interpret(study, 'reasoning_on')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'reasoning_on')))


recorded Dell measurement reasoning_on: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T033328_571911Z.json](../experiment-tracker/mvp4/nemotron/20261107T033328_571911Z.json)

**Prediction:** Reasoning/output budgets change generation work and may affect task completion.

**Observation:** Application result match: 2/4; SQL executed: 2/4. Request E2E p50/p95: 9.104/16.138 s; aggregate output: 43.133 tok/s.

**Explanation:** Thoughts consume output tokens; TTFT includes thoughts, and truncated answers remain failures.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Use the smallest budget that preserves required task quality; do not optimize output tok/s alone.

#### output 256


In [13]:
result = study.run('output_256')
print(result.get('_recorded', {}).get('label', study.mode), 'output_256:', result['status'])
display(Markdown(interpret(study, 'output_256')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'output_256')))


recorded Dell measurement output_256: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T034418_092327Z.json](../experiment-tracker/mvp4/nemotron/20261107T034418_092327Z.json)

**Prediction:** Reasoning/output budgets change generation work and may affect task completion.

**Observation:** Application result match: 2/4; SQL executed: 2/4. Request E2E p50/p95: 4.265/4.918 s; aggregate output: 36.422 tok/s.

**Explanation:** Thoughts consume output tokens; TTFT includes thoughts, and truncated answers remain failures.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Use the smallest budget that preserves required task quality; do not optimize output tok/s alone.

#### output 2048


In [14]:
result = study.run('output_2048')
print(result.get('_recorded', {}).get('label', study.mode), 'output_2048:', result['status'])
display(Markdown(interpret(study, 'output_2048')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'output_2048')))


recorded Dell measurement output_2048: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nemotron/20261107T035411_952257Z.json](../experiment-tracker/mvp4/nemotron/20261107T035411_952257Z.json)

**Prediction:** Reasoning/output budgets change generation work and may affect task completion.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 4.268/4.914 s; aggregate output: 36.491 tok/s.

**Explanation:** Thoughts consume output tokens; TTFT includes thoughts, and truncated answers remain failures.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Use the smallest budget that preserves required task quality; do not optimize output tok/s alone.

### Optional precision stretch: compatible checkpoint required

Each row is its own controlled comparison. No cumulative best configuration was measured.


#### awq stretch


In [15]:
result = study.run('awq_stretch')
print(result.get('_recorded', {}).get('label', study.mode), 'awq_stretch:', result['status'])
display(Markdown(interpret(study, 'awq_stretch')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'awq_stretch')))


replay awq_stretch: unavailable


**Prediction:** Four-bit weights reduce weight storage; latency and SQL quality may change in either direction.

**Observation:** No such measurement was recorded in the Dell archive; no values estimated.

**Explanation:** Quantization changes weight representation and kernel behavior; caching addresses prefix reuse.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Benchmark compatible AWQ when weight capacity matters; no native VRAM reduction is established here.

## 5. Put performance beside quality

TTFT is first generated content. Request E2E measures generation/transport; application E2E
includes planning, validation, execution and repair. Cold/warm cache phases are separate from
supplemental warmed summaries. Semantic review remains unreviewed. Native VRAM telemetry
failed and stays missing; no memory-reduction conclusion is supported by those runs.


In [16]:
display(Markdown(study.report()))
display(Markdown(study.cache_report()))


| Configuration/status | SQL | Result match | Semantic review | Prompt tokens | TTFT p50 s | Latency p50/p95 s | tok/s | Peak memory |
|---|---|---|---|---|---|---|---|---|
| MVP4/Nemotron/fp16_baseline · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | — | 1.05 | 4.25/4.89 | 36.62 | — |
| MVP4/Nemotron/prefix_cache · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | — | 0.32 | 3.58/4.21 | 43.79 | — |
| MVP4/Nemotron/schema_retrieved · recorded Dell measurement | 1/4 | 1/4 | Unreviewed; 4 total | — | 0.20 | 2.59/3.55 | 46.30 | — |
| MVP4/Nemotron/concurrency_2 · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | — | 1.10 | 5.36/7.01 | 56.24 | — |
| MVP4/Nemotron/concurrency_4 · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | — | 1.11 | 7.70/10.56 | 76.88 | — |
| MVP4/Nemotron/batch_capacity_1 · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | — | 13.39 | 16.70/16.75 | 36.38 | — |
| MVP4/Nemotron/reasoning_on · recorded Dell measurement | 2/4 | 2/4 | Unreviewed; 4 total | — | 1.06 | 9.10/16.14 | 43.13 | — |
| MVP4/Nemotron/output_256 · recorded Dell measurement | 2/4 | 2/4 | Unreviewed; 4 total | — | 1.06 | 4.27/4.92 | 36.42 | — |
| MVP4/Nemotron/output_2048 · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | — | 1.05 | 4.27/4.91 | 36.49 | — |
| nemotron/awq_stretch · unavailable | — | — | — | — | — | — | — | — |

| Experiment | Application E2E p50/p95 s | Request TTFT p95 s |
|---|---|---|
| fp16_baseline · recorded Dell measurement | 10.711/17.867 | 1.064 |
| prefix_cache · recorded Dell measurement | 8.828/15.832 | 0.327 |
| schema_retrieved · recorded Dell measurement | 8.117/9.086 | 0.267 |
| concurrency_2 · recorded Dell measurement | 10.199/17.928 | 1.677 |
| concurrency_4 · recorded Dell measurement | 10.220/17.924 | 3.390 |
| batch_capacity_1 · recorded Dell measurement | 10.255/17.978 | 14.526 |
| reasoning_on · recorded Dell measurement | 24.388/37.500 | 1.072 |
| output_256 · recorded Dell measurement | 8.861/11.656 | 1.067 |
| output_2048 · recorded Dell measurement | 10.187/17.951 | 1.064 |
| awq_stretch | —/— | — |

| Variant | Phase | SQL match | Semantic | TTFT p50/p95 s | E2E p50/p95 s | TPOT p50 s | tok/s | Peak VRAM GiB |
|---|---|---|---|---|---|---|---|---|
| prefix_cache · recorded Dell measurement | cold | 15/20 | unreviewed | 1.101/1.109 | 4.327/4.968 | 0.021 | 35.945 | — |
| prefix_cache · recorded Dell measurement | warm | 15/20 | unreviewed | 0.300/0.305 | 3.529/4.179 | 0.021 | 44.274 | — |

### Recorded evidence figure

![Recorded Dell measurement](../tutorial/figures/02_concurrency.png)

This figure always describes the archived Dell runs, including when the notebook is in live
mode. Fresh measurements appear in the tables above. [Figure sources](../tutorial/figures/figure_data.json)
· [reproduction script](../scripts/tutorial/render_findings.py).


## 6. Finish, preserve and continue

Save with Ctrl+S. Replay changes only the notebook display; it does not rewrite measurements.
Live adds unique records and owns only the processes it starts.
Native servers stop after each row; the final cleanup cell is safe to repeat.
Interrupted/failed operations clean up owned servers; a kernel/OS crash requires explicit
cleanup using the saved process group.


In [17]:
print(study.close())


Study servers stopped. Saved records and notebook outputs are retained.


Continue to [the next chapter](03_qwen_nim.ipynb), or read:

- [What should I choose?](../tutorial/RESULTS.md#what-should-i-choose) — concise results and deployment decisions.
- [Troubleshooting case studies](../tutorial/CASE_STUDIES.md).
- [Four-case SQL review packet](../tutorial/SQL_REVIEW.md) — human review pending.
- [Fresh GPU validation checklist](../tutorial/GPU_VALIDATION.md).
- [Original executed notebook](history/dell-2026-11-07/local%20AI%20optimization%20-%20Nemotron.ipynb) and [checksum manifest](history/dell-2026-11-07/MANIFEST.json).

Historical iteration labels and source lineage are preserved in the evidence, not used as
the learner-facing chapter structure.
